# 🎯 [TRN01] 드론 시뮬레이터 비행로그 기반 숙련도 분석기
### Google Colab 실습 환경 (교육·훈련 / Level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/TRN01/trn01_m05_student.ipynb)

---

## 📌 1. 프로젝트 개요 및 학습 목표

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 교육·훈련 (Level 3) |
| **학습 소요 시간** | 10개 미션 (5일 / 총 27.5시간) |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 학습 목표 및 대상자
러시아-우크라이나 전쟁에서 소형 드론 조종 인력을 단기간에 양성·평가해야 했던 전장 요구를 배경으로, 교육용 가상 비행로그를 직접 생성·분석하며 드론 숙련도 평가 파이프라인 전 주기를 설계·구현하는 프로젝트다. 최종 목표는 교관 평가와 정합성을 갖춘 객관적 지표와 개인별 피드백을 산출하되, 등급과 재교육 결정은 교관이 근거를 기록해 확정하는 신뢰 가능한 분석기를 완성하는 것이다.

#### 본 프로젝트를 통해 익히는 핵심 역량

- **전술 시계열 데이터 설계 및 정제 역량:** 비행로그 스키마와 보안(가명화) 규칙을 정의하고, PWM 정규화·Hampel 필터·보간으로 센서 잡음과 단절을 복구하는 실무 기법을 익힌다.
- **도메인 기반 숙련도 모델링 및 한계 분석 역량:** 경로·호버링 Baseline 채점부터 장애물 회피·비상 착륙 반응 기반 정밀 평가 엔진까지 단계적으로 고도화하며, 혼동표·오탐 분석으로 모델의 사각지대를 찾아내는 방법을 습득한다.
- **통계적 신뢰도 검증 및 조작 습관 진단 역량:** 단발성 점수의 오류를 줄이기 위해 다회차 비행 묶음 평가와 부트스트랩 신뢰구간을 적용하고, 급조작·과보정 등 개인별 습관을 정량적으로 판별하는 원리를 익힌다.
- **HITL(인간-중심) 거버넌스 및 시스템 복원력 설계 역량:** 저신뢰·고위험 사례 우선 검토, 2단계 교관 확인, 해시 체인 감사 로그로 AI 책임 분계를 구현하고, 대량 배치 스트레스 시험과 AAR 재튜닝으로 운용 안정성을 확보하는 역량을 기른다.

#### 추천 학습 대상자 (페르소나)

- **드론·무인기 교육훈련 교관 및 평가 담당자:** 시뮬레이터 비행로그를 정량 지표로 해석하고, 교관의 경험적 평가와 AI 분석 결과를 결합해 개인별 맞춤 피드백·재교육 기준을 정립하고자 하는 현장 실무자.
- **국방 AI·데이터 분석 실무자 및 개발자:** Python 기본 문법을 바탕으로 시계열 노이즈 전처리, 다중 지표 채점 모델링, 부트스트랩 신뢰도 산출, HITL 승인 워크플로우까지 End-to-End 분석 파이프라인을 직접 구현하려는 개발자.
- **무인체계 시뮬레이터·훈련체계 기획 및 품질 검증 담당자:** 로그 스키마 표준화, 대량 배치 스트레스 테스트, 사후강평(AAR) 기반 파라미터 재튜닝, 감사 로그 거버넌스 등 신뢰 가능한 AI 보조 평가 체계의 설계 기준을 익히려는 엔지니어.


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 05. 조종자별 조작 습관(급조작, 과보정) 분석 및 신뢰도 등급 산출 모듈 개발
**소요시간**: 약 3.0시간 | **필수 산출물**: src/verification/trn01_confidence_engine.py (필수)

### 📋 미션 개요 (Overview)
단일 비행 점수만으로 재교육 여부를 판단하면 일시적 환경 변수나 우연한 실수로 오지정이 발생할 수 있다. 본 미션에서는 동일 조건 조종자 대비 스틱 변화율과 회피 후 초과 이동으로 급조작·과보정 습관을 판별하고, 조종자당 6회 비행 점수를 묶어 부트스트랩 기반 등급 신뢰도(High/Medium/Low)를 산출하는 모듈(`trn01_confidence_engine.py`)을 개발한다. 기존 1회 Baseline 방식과 재교육 오지정률·재현율을 비교해 다회차 평가의 타당성을 검증한다.

### 🎓 학습 목표 (Objectives)
단일 비행 평가의 변동성을 극복하기 위해 **다회차 비행 데이터를 집계해 조작 습관을 판별하고 부트스트랩으로 예측 불확실성을 정량화하는 통계적 검증 역량**을 익힌다. 동일 조건 기준 집단 대비 급조작·과보정 판별 기준을 세우고, 6회 비행 묶음 등급과 신뢰등급(High/Medium/Low)을 산출해 1회 Baseline 대비 재교육 오지정률·재현율 및 ROC-AUC 개선 효과를 검증하는 방법을 배운다.

### 🛠️ 단계별 수행 및 AI 활용 가이드
#### Step 1. 미션5 코드를 실행해 습관 판정 정확도를 확인한다.
#### 이 단계의 역할
미션5 코드를 실행해 습관 판정 정확도를 확인한다.

#### 해야 할 일

- 미션5 코드를 실행해 습관 판정 정확도를 확인한다.에 필요한 데이터셋과 입력 파라미터를 준비한다.
- 정답 코드 및 모듈을 실행하여 중간 출력 결과를 도출한다.
- 출력된 지표와 로그가 기준 조건을 만족하는지 대조·검증한다.

#### 핵심 확인 지표
> 오지정률 71.0% 감축(목표 40%), 등급 일관성 97.6% 습관 판정 정확도 급조작 95%·과보정 88%

#### 학습자가 제출해야 하는 내용

- 미션5 코드를 실행해 습관 판정 정확도를 확인한다. 실행 결과 및 검증 로그

#### AI 활용 가이드 (프롬프트 예시 & 검증 포인트)
**AI 질의 예시**

> 동일 조건의 기준 집단 대비 스틱 변화율과 회피 후 초과 이동량을 계산해 조종자별 급조작·과보정 습관을 판별하는 로직과 정확도 검증 코드를 작성해줘.
**AI 응답 검증 포인트**

- 습관 레이블이 부여된 가상 조종자 데이터에서 급조작·과보정 판별 정확도와 오탐률을 확인
- 조건(주간/야간/돌풍)을 섞지 않고 동일 조건 내 상대 비교가 이루어졌는지 검증

#### Step 2. 기존 방식과 엔진의 재교육 지정 오지정률·재현율을 비교한다.
#### 이 단계의 역할
기존 방식과 엔진의 재교육 지정 오지정률·재현율을 비교한다.

#### 해야 할 일

- 기존 방식과 엔진의 재교육 지정 오지정률·재현율을 비교한다.에 필요한 데이터셋과 입력 파라미터를 준비한다.
- 정답 코드 및 모듈을 실행하여 중간 출력 결과를 도출한다.
- 출력된 지표와 로그가 기준 조건을 만족하는지 대조·검증한다.

#### 핵심 확인 지표
> 오지정률 71.0% 감축(목표 40%), 등급 일관성 97.6% 습관 판정 정확도 급조작 95%·과보정 88%

#### 학습자가 제출해야 하는 내용

- 기존 방식과 엔진의 재교육 지정 오지정률·재현율을 비교한다. 실행 결과 및 검증 로그

#### AI 활용 가이드 (프롬프트 예시 & 검증 포인트)
**AI 질의 예시**

> 조종자당 6회 비행을 묶어 등급을 내는 방식과 기존 1회 Baseline 점수 방식의 재교육 대상 오지정률·재현율, ROC-AUC를 비교하는 표를 생성해줘.
**AI 응답 검증 포인트**

- 다회차 묶음 방식이 1회 평가 대비 오지정률을 유의미하게 낮추면서도 재현율을 유지하는지 확인
- ROC-AUC 계산 시 양성 클래스(재교육 필요 C등급) 정의가 일관적인지 점검

#### Step 3. 기준점수 민감도와 등급 일관성, 신뢰등급 분포를 해석한다.
#### 이 단계의 역할
기준점수 민감도와 등급 일관성, 신뢰등급 분포를 해석한다.

#### 해야 할 일

- 기준점수 민감도와 등급 일관성, 신뢰등급 분포를 해석한다.에 필요한 데이터셋과 입력 파라미터를 준비한다.
- 정답 코드 및 모듈을 실행하여 중간 출력 결과를 도출한다.
- 출력된 지표와 로그가 기준 조건을 만족하는지 대조·검증한다.

#### 핵심 확인 지표
> 오지정률 71.0% 감축(목표 40%), 등급 일관성 97.6% 습관 판정 정확도 급조작 95%·과보정 88%

#### 학습자가 제출해야 하는 내용

- 기준점수 민감도와 등급 일관성, 신뢰등급 분포를 해석한다. 실행 결과 및 검증 로그

#### AI 활용 가이드 (프롬프트 예시 & 검증 포인트)
**AI 질의 예시**

> 부트스트랩 리샘플링으로 조종자별 동일 등급 유지 확률을 계산해 신뢰등급(High/Medium/Low)을 부여하고, 기준점수(cut-off) 민감도 분석 결과를 해석해줘.
**AI 응답 검증 포인트**

- 경계 점수(예: B/C 경계)에 있는 조종자가 Low 신뢰등급으로 적절히 분류되는지 확인
- 부트스트랩 반복 횟수(예: 500~1000회)가 결과 재현성(고정 시드)을 보장하는지 점검

#### Step 4. 도전 과제 및 파라미터 튜닝 최적화
#### 이 단계의 역할
기본 파이프라인 수행 후 심화 튜닝 및 조건 변경을 통해 최적화 역량을 다진다.

#### 해야 할 일
재교육 기준점수를 55에서 60으로 올리면 지정 인원과 오지정률이 어떻게 바뀌는지 확인하고, 교육 자원과 안전을 고려한 기준을 토의한다.

#### 학습자가 제출해야 하는 내용

- 파라미터 변경 전후 비교 분석 메모 및 최종 최적화 결과

#### AI 활용 가이드 (프롬프트 예시 & 검증 포인트)
**AI 질의 예시**

> 조종자당 비행 횟수를 6회에서 3회 또는 4회로 줄였을 때 등급 일관성과 Low 신뢰등급 비율이 어떻게 변하는지 시뮬레이션해줘.
**AI 응답 검증 포인트**

- 비행 횟수 축소에 따른 불확실성 증가 폭이 신뢰등급 분포에 정확히 반영되는지 확인
- 최소 몇 회 이상의 비행이 있어야 실무 재교육 판정에 활용 가능한지 근거를 도출


In [ ]:
"""조종자별 여러 비행의 조작 습관(급조작=스틱 변화율, 과보정=회피 후 초과 이동)을 분석하고, 비행 점수를 묶어 숙련도 등급과 신뢰도(부트스트랩 등급 확률)를 산출한다. 비행 1회로 '재교육 대상'을 지정하던 방식 대비 오지정률(FAR) 감축, ROC-AUC, 임계값 민감도, 등급 일관성을 검증한다."""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.metrics import roc_auc_score

# ---------------------------------------------------------------------------
# [교육용가상자료] 드론 조종 시뮬레이터 비행로그 생성기 (실제 훈련생·장비 정보 없음, 조종자는 가명 P001~)
# 비행 1회 = 120초(10Hz): 경로 비행(0~60초) -> 호버링(60~85초) -> 돌발 장애물 회피(85초) -> 비상 착륙(100초 지시)
# 조종자마다 숨은 숙련도(skill)와 조작 습관(급조작 jerky, 과보정 overcorr)이 있고, 교관 점수는 이를 반영해 만든다.
# ---------------------------------------------------------------------------
DT, STEPS = 0.1, 1200
T = np.arange(STEPS) * DT
COND = {"day": dict(wind=0.4, gust=0.0, gps=0.25, delay=0.0), "night": dict(wind=0.4, gust=0.0, gps=0.5, delay=0.15),
        "gust": dict(wind=1.2, gust=2.5, gps=0.25, delay=0.0)}
OBST_T, LAND_T = 85.0, 100.0


def make_pilots(n=40, seed=7):
    rng = np.random.default_rng(seed)
    skill = rng.beta(2.2, 2.0, n)
    return pd.DataFrame({"pilot": [f"P{i + 1:03d}" for i in range(n)], "skill": skill,
                         "jerky": rng.random(n) < 0.15 + 0.35 * (1 - skill), "overcorr": rng.random(n) < 0.15 + 0.35 * (1 - skill)})


def target_path(t):
    """사각형 경로(40x30m, 고도 20m)를 3m/s로 비행 -> 60초 지점에서 호버링 -> 착륙 지시 후 고도 0."""
    # [TODO : 학생 실습 5] 조종자별 조작 습관(급조작, 과보정) 분석 및 신뢰도 등급 산출 모듈 개발 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def fly(pilots, cond="day", seed=0, delay_extra=0.0, obstacle_side=None):
    """여러 조종자의 비행 1회를 동시에 모의한다(벡터화). 반환: 로그 배열 dict, 비행별 정답 정보 DataFrame."""
    # [TODO : 학생 실습 5] 조종자별 조작 습관(급조작, 과보정) 분석 및 신뢰도 등급 산출 모듈 개발 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def one(log, j):
    """여러 조종자 로그에서 j번째 조종자 1명의 로그(DataFrame)를 꺼낸다."""
    return pd.DataFrame({k: (v if v.ndim == 1 else v[:, j]) for k, v in log.items()})
# ---------------------------------------------------------------------------

# ---- 비행로그 특징 추출 공통 함수(여러 조종자 로그를 한 번에 계산) ----
from scipy.ndimage import uniform_filter1d

BASE_F = ["path_rmse", "hover_drift", "hover_std", "alt_err"]
ADV_F = BASE_F + ["obs_rt", "obs_lateral", "touch_vz", "att_exceed", "jerk", "overshoot", "n_risk"]
RULES = {"near_collision": ("obs_lateral", "<", 2.5), "hard_landing": ("touch_vz", ">", 2.0), "attitude_exceed": ("att_exceed", ">=", 10)}


def detect(df):
    """위험 조작 사건 탐지 규칙: 장애물 2.5초 후 회피 거리 2.5m 미만, 접지 속도 2.0m/s 초과, 자세 35도 초과 1초 이상."""
    out = {}
    for ev, (f, op, th) in RULES.items():
        out[ev] = df[f] < th if op == "<" else (df[f] > th if op == ">" else df[f] >= th)
    return pd.DataFrame(out)


def features(log):
    """경로 이탈도·호버링 안정성(기본) + 장애물 반응·착륙 속도·자세 한계·급조작·과보정(고도화) 특징."""
    X, Y, A = (uniform_filter1d(log[k], 9, axis=0) for k in ("x", "y", "alt"))       # GPS 잡음 완화(이동평균 0.9초)
    h = np.hypot(X - log["tx"], Y - log["ty"])
    cr, hv = slice(0, 600), slice(620, 850)
    k0, k1 = int(OBST_T / DT), int((OBST_T + 2.5) / DT)
    side = np.sign(log["ty"][k0 + 1] - log["ty"][k0 - 1])
    moved = (log["stick_roll"][k0:k0 + 40] * side) > 0.3
    rt = np.where(moved.any(0), moved.argmax(0) * DT, 4.0)
    vz = -np.gradient(A, DT, axis=0)
    land = np.argmax((A < 0.4) & (T[:, None] >= LAND_T), axis=0)
    touch = np.array([vz[max(i - 6, 0):i - 2, j].mean() if i > 6 else 3.0 for j, i in enumerate(land)])
    ds = np.abs(np.diff(np.stack([log["stick_roll"], log["stick_pitch"]]), axis=1)) / DT
    return pd.DataFrame({
        "path_rmse": np.sqrt((h[cr] ** 2).mean(0)), "hover_drift": h[hv].mean(0), "hover_std": np.hypot(X[hv].std(0), Y[hv].std(0)),
        "alt_err": np.abs(A[:850] - log["talt"][:850]).mean(0),
        "obs_rt": rt, "obs_lateral": np.abs(Y[k1] - Y[k0]), "touch_vz": touch,
        "att_exceed": (np.maximum(np.abs(log["roll_deg"]), np.abs(log["pitch_deg"])) > 35).sum(0),
        "jerk": np.percentile(ds, 95, axis=(0, 1)),
        "overshoot": np.clip(np.abs(Y[k0:k0 + 120] - Y[k0]).max(0) - 6.0, 0, None)}).pipe(
        lambda d: d.assign(n_risk=detect(d).sum(axis=1)))                                     # overshoot: 회피 후 목표 6m를 넘어선 거리(과보정)


def make_dataset(n_pilots=40, seed=7, conds=("day", "night", "gust"), flights=1):
    """조종자 n명 x 조건 x 반복 비행 -> 비행별 특징 + 정답(교관 점수·등급·위험 사건) 표."""
    pilots = make_pilots(n_pilots, seed)
    out = []
    for c in conds:
        for f in range(flights):
            log, tr = fly(pilots, c, seed=seed * 100 + list(COND).index(c) * 10 + f)
            out.append(pd.concat([tr.reset_index(drop=True), features(log)], axis=1).assign(flight=f))
    return pd.concat(out, ignore_index=True)


def cond_z(df, ref, cols):
    """조건별 기준 집단(ref)의 평균·표준편차로 표준화: 돌풍처럼 어려운 조건에서 받는 불이익을 줄인다."""
    z = df.copy()
    z[cols] = z[cols].astype(float)
    for c in df["cond"].unique():
        r = ref[ref["cond"] == c][cols]
        z.loc[df["cond"] == c, cols] = (df.loc[df["cond"] == c, cols] - r.mean()) / (r.std() + 1e-9)
    return z


def to_grade(score):
    return np.where(score >= 75, "A", np.where(score >= 55, "B", "C"))

train = make_dataset(40, seed=7)
eng = Ridge(alpha=1.0).fit(cond_z(train, train, ADV_F)[ADV_F], train["instructor"])
base = Ridge(alpha=1.0).fit(cond_z(train, train, BASE_F)[BASE_F], train["instructor"])     # 미션 3 Baseline 채점식
zt = cond_z(train, train, ADV_F)                                         # 습관 판정 기준: 같은 조건에서 습관 없는 조종자의 상위 10%
jerk_th = zt.loc[~train["jerky"], "jerk"].quantile(0.9)
over_th = zt.loc[~train["overcorr"], "overshoot"].quantile(0.9)

# [교육용가상자료] 평가 조종자 80명 x 비행 6회(주간·야간·돌풍 각 2회)
test = make_dataset(80, seed=9, flights=2)
test["score"] = np.clip(eng.predict(cond_z(test, train, ADV_F)[ADV_F]), 0, 100)
ze = cond_z(test, train, ADV_F)
test["jerk_z"], test["over_z"] = ze["jerk"], ze["overshoot"]
test["base"] = np.clip(base.predict(cond_z(test, train, BASE_F)[BASE_F]), 0, 100)
rng = np.random.default_rng(0)
rows = []
for pid, g in test.groupby("pilot"):
    boot = [to_grade(rng.choice(g["score"].values, len(g)).mean()).item() for _ in range(200)]
    grade = to_grade(g["score"].mean()).item()
    conf = boot.count(grade) / len(boot)
    rows.append({"pilot": pid, "점수": g["score"].mean(), "점수SD": g["score"].std(), "등급": grade, "신뢰도": conf,
                 "신뢰등급": "High" if conf >= 0.9 else ("Medium" if conf >= 0.7 else "Low"),
                 "급조작": (g["jerk_z"] > jerk_th).mean() >= 0.5, "과보정": (g["over_z"] > over_th).mean() >= 0.5,
                 "실제급조작": g["jerky"].iloc[0], "실제과보정": g["overcorr"].iloc[0],
                 "교관등급": to_grade(g["instructor"].mean()).item(), "단일비행점수": g["base"].iloc[int(rng.integers(len(g)))]})
P = pd.DataFrame(rows)
print(f"습관 판정 정확도: 급조작 {(P['급조작'] == P['실제급조작']).mean():.0%}, 과보정 {(P['과보정'] == P['실제과보정']).mean():.0%} (조종자 {len(P)}명)")

# 재교육 대상(C) 지정: 기존(무작위 비행 1회의 Baseline 점수 55 미만) vs 엔진(6회 평균 C + 신뢰도 Medium 이상, Low는 교관 확인으로 보냄)
need = P["교관등급"] == "C"
old = P["단일비행점수"] < 55
new = (P["등급"] == "C") & (P["신뢰등급"] != "Low")
review = (P["등급"] == "C") & (P["신뢰등급"] == "Low")


def far_rec(flag):
    return (flag & ~need).sum() / max(flag.sum(), 1), (flag & need).sum() / max(need.sum(), 1), int(flag.sum())


f0, r0, n0 = far_rec(old)
f1, r1, n1 = far_rec(new)
print(pd.DataFrame([{"방식": "기존(비행 1회 Baseline<55)", "지정": n0, "오지정률": round(f0, 3), "재현율": round(r0, 3)},
                    {"방식": "엔진(6회 평균 C, 신뢰 Medium 이상)", "지정": n1, "오지정률": round(f1, 3), "재현율": round(r1, 3)}]).to_string(index=False))
cut = 1 - f1 / max(f0, 1e-9)
print(f"오지정률(FAR) 감축 {cut:.1%} (목표 40% 이상: {'달성' if cut >= 0.4 else '미달'}), 신뢰도 Low로 교관 확인에 넘긴 조종자 {int(review.sum())}명")
print(f"ROC-AUC(재교육 필요 판별, 평균 점수 기준) {roc_auc_score(need, -P['점수']):.3f}")

print("\n기준점수  지정  오지정률  재현율")
for th in [50, 55, 60, 65]:
    f, r, n = far_rec(P["점수"] < th)
    print(f"  {th}      {n:3d}   {f:.3f}   {r:.3f}")

# 등급 일관성: 비행 6회 중 5회만 무작위로 써도 같은 등급이 나오는 비율
cons = []
for pid, g in test.groupby("pilot"):
    full = to_grade(g["score"].mean()).item()
    cons += [to_grade(g["score"].sample(5, random_state=k).mean()).item() == full for k in range(20)]
print(f"\n등급 일관성 {np.mean(cons):.1%} (목표 90% 이상: {'달성' if np.mean(cons) >= 0.9 else '미달'})")
print("[신뢰등급 분포]", P["신뢰등급"].value_counts().to_dict(), "/ 등급 일치(교관 대비)", f"{(P['등급'] == P['교관등급']).mean():.0%}")
print("신뢰등급 기준: High(부트스트랩 동일 등급 90% 이상)=결과 통보, Medium(70~90%)=교관 검토 후 통보, Low=추가 비행 후 재평가")

P = P.sort_values("점수")
plt.figure(figsize=(10, 4))
col = P["신뢰등급"].map({"High": "tab:green", "Medium": "tab:orange", "Low": "tab:red"})
plt.errorbar(range(len(P)), P["점수"], yerr=P["점수SD"] / np.sqrt(6) * 2, fmt="none", ecolor="gray")
plt.scatter(range(len(P)), P["점수"], c=col)
plt.axhline(55, ls="--", c="k"); plt.axhline(75, ls="--", c="k")
plt.xticks(range(len(P)), P["pilot"], rotation=90, fontsize=6); plt.ylabel("mean skill score (+-2 SE)")
plt.title("Pilot skill score with uncertainty (color = confidence High/Medium/Low)"); plt.tight_layout(); plt.show()

# TODO: 조작 습관·신뢰도 모듈을 src/verification/trn01_confidence_engine.py 로 저장하세요.


<details>
<summary><b>💡 [참고] 미션 05 예상 실행 결과 (Output)</b></summary>

```text
습관 판정 정확도: 급조작 95%, 과보정 88% (조종자 80명)
                       방식  지정  오지정률   재현율
    기존(비행 1회 Baseline<55)  27 0.111 0.750
엔진(6회 평균 C, 신뢰 Medium 이상)  31 0.032 0.938
오지정률(FAR) 감축 71.0% (목표 40% 이상: 달성), 신뢰도 Low로 교관 확인에 넘긴 조종자 1명
ROC-AUC(재교육 필요 판별, 평균 점수 기준) 0.997

기준점수  지정  오지정률  재현율
  50       22   0.000   0.688
  55       32   0.031   0.969
  60       43   0.256   1.000
  65       53   0.396   1.000

등급 일관성 97.6% (목표 90% 이상: 달성)
[신뢰등급 분포] {'High': 66, 'Medium': 9, 'Low': 5} / 등급 일치(교관 대비) 98%
신뢰등급 기준: High(부트스트랩 동일 등급 90% 이상)=결과 통보, Medium(70~90%)=교관 검토 후 통보, Low=추가 비행 후 재평가
```
</details>
